# Демонстрация: сравнение PEFT-методов для анализа тональности

Технологическая практика — программная реализация и экспериментальное исследование параметро-эффективных методов дообучения (PEFT) трансформерных моделей.



## 1. Подготовка: код проекта и зависимости



In [ ]:

!pip install -q -r requirements.txt
print("Зависимости установлены.")

## 2. Проверка окружения

In [ ]:
import torch
import config

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Память GPU: %.1f ГБ" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
else:
    print("ВНИМАНИЕ: GPU не найден — обучение будет очень медленным. Включите GPU в настройках среды.")

print("\nБазовая модель:", config.MODEL_NAME)
print("Датасеты:", list(config.DATASETS.keys()))
print("Методы:", list(config.METHODS.keys()))

## 3. ТЕСТ

Прогоняем все методы на одном датасете (SST-2) с маленькими подвыборками — чтобы убедиться, что весь pipeline работает и каждый метод действительно обучается.

In [ ]:
from main import run

results_quick = run(datasets=["SST-2"], quick=True)

## 4. Полный эксперимент

Оба датасета (IMDb + SST-2), все методы, полные подвыборки из `config.py`.
Флаг `save_adapters=True` дополнительно сохранит обученные адаптеры в папку `adapters/`.

In [ ]:
results = run(quick=False, save_adapters=True)

## 6. Демонстрация загрузки сохранённого адаптера


In [ ]:
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

adapter_dir = "adapters/LoRA_r8"  # путь к сохранённому адаптеру LoRA (r=8)

if os.path.exists(adapter_dir):
    tok = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    base = AutoModelForSequenceClassification.from_pretrained(config.MODEL_NAME, num_labels=config.NUM_LABELS)
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()

    texts = ["This movie was absolutely wonderful and touching.",
             "A complete waste of time, terrible acting."]
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        preds = model(**enc).logits.argmax(-1).tolist()
    for t, p in zip(texts, preds):
        print(f"[{'POSITIVE' if p == 1 else 'NEGATIVE'}] {t}")
else:
    print("Адаптер не найден — сначала запустите ячейку 4 с save_adapters=True.")